# LangGraph

### A computational graph for cognition

- an extension of **LangChain**
- where **nodes** *think*,
- **edges** *decide*,
- and **state** *remembers*.

### Install necessary libraries and packages

In [1]:
!pip install langgraph langchain openai langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.0/513.0 kB 14.2 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28


### Setup API key to use OpenAI services

- use Colab secret if it is defined
- otherwise look for the OS environment variable
- if that fails, ask the user to input their key

In [2]:
import os
from getpass import getpass

from google.colab import userdata
from langchain_openai import ChatOpenAI

if "OPENAI_API_KEY" not in os.environ:
    # Try to get the API key from Colab secrets
    colab_secret = userdata.get("OPENAI_API_KEY")
    if colab_secret:
        os.environ["OPENAI_API_KEY"] = colab_secret
    else:
        # If not in secrets, prompt the user
        os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

model = ChatOpenAI(model="gpt-4.1")

### Build on top of LangChain

- given the text of a prompt (template)
- return the equivalent short chain

In [3]:
def build_chain(prompt_text):
    prompt = ChatPromptTemplate.from_template(prompt_text)
    chain = prompt | model | StrOutputParser()
    return chain

### Simple LLM Node

- a trivial graph
- just *call the LLM*
- nodes can encapsulate LLM calls as pure functions over state
- This is the atomic unit of cognition

In [4]:
from typing import TypedDict

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI


class SimpleState(TypedDict):
    prompt: str
    response: str


def call_llm(state):
    result = build_chain(state["prompt"]).invoke({})
    return {"response": result}

### Simple LLM Chain (LangChain equivalent)

#### Contrast with LangChain:

- Same functionality

- But now explicitly graph-structured

In [5]:
from langgraph.graph import END, START, StateGraph

graph = StateGraph(SimpleState)

graph.add_node("llm", call_llm)

graph.set_entry_point("llm")
graph.add_edge("llm", END)

simple_app = graph.compile()

simple_app.invoke({"prompt": "Explain transformers in a haiku"})

{'prompt': 'Explain transformers in a haiku',
 'response': 'Words pass through deep nets,  \nAttention learns what to hold—  \nMeaning flows, transformed.'}

### Reflection for Joke Generation (ReAct pattern)

- ask the LLM to critique itself
- Generate a joke and then reflect on it
- Re-generate the joke until it passes muster

In [6]:
class JokeState(TypedDict):
    topic: str
    joke: str
    best: str
    feedback: str
    viewpointPRO: str
    viewpointANTI: str
    attempts: int


max_attempts = 10


def generate(state):
    print(f"Attempt #{state['attempts']}")
    prompt = f"Generate a setup/punchline joke about this topic: {state['topic']}. Don't be afraid to be mean. Return just the joke."
    for _ in range(max_attempts):
        joke = build_chain(prompt).invoke({})
        if joke and joke.strip():
            return {"joke": joke.replace("\n", "\t"), "attempts": state["attempts"] + 1}

    return {"joke": "No joke", "attempts": state["attempts"] + 1}


def critique(state):
    if not state["joke"].strip():
        print("   Candidate [unfunny]: EMPTY")
        return {"feedback": "unfunny"}
    chain = build_chain(
        f"Is this a funny joke or a dull one? Just say 'funny' or 'unfunny': {state['joke']}"
    )
    feedback = chain.invoke({}).strip().lower()
    print(f"   Candidate [{feedback}]: {state['joke']}")
    if feedback == "funny" and "best" in state:
        return {"feedback": feedback, "best": state["joke"]}
    return {"feedback": feedback}


def offendPRO(state):
    chain = build_chain(
        f"From the perspective of a fan of {state['topic']} is this an offensive joke or an inoffensive one? Just say 'offensive' or 'inoffensive': {state['joke']}"
    )
    viewpoint = chain.invoke({}).strip().lower()
    print(f"   PRO: {viewpoint}")
    if viewpoint == "offensive":
        return {"viewpointPRO": viewpoint, "best": state["joke"]}
    return {"viewpointPRO": viewpoint}


def offendANTI(state):
    chain = build_chain(
        f"From the perspective of a critic of {state['topic']} is this an offensive joke or an inoffensive one? Just say 'offensive' or 'inoffensive': {state['joke']}"
    )
    viewpoint = chain.invoke({}).strip().lower()
    print(f"  ANTI: {viewpoint}")
    if viewpoint == "inoffensive":
        return {"viewpointANTI": viewpoint, "best": state["joke"]}
    return {"viewpointANTI": viewpoint}

### Build the graph for our app

- and invoke it with a user prompt

- This code implements the **ReAct** (**Re**asoning + **Act**ion) pattern

- Cycles = iterative reasoning

In [7]:
graph = StateGraph(JokeState)

graph.add_node("generate", generate)
graph.add_node("critique", critique)
graph.add_node("offendPRO", offendPRO)
graph.add_node("offendANTI", offendANTI)

graph.set_entry_point("generate")

graph.add_edge("generate", "critique")

graph.add_conditional_edges(
    "critique",
    lambda s: (
        "max"
        if s["attempts"] > max_attempts
        else ("retry" if "unfunny" in s["feedback"] else "done")
    ),
    {"retry": "generate", "done": "offendPRO", "max": END},
)

graph.add_conditional_edges(
    "offendPRO",
    lambda s: (
        "retry"
        if "inoffensive" in s["viewpointPRO"] and s["attempts"] < max_attempts
        else "done"
    ),
    {"retry": "generate", "done": "offendANTI"},
)

graph.add_conditional_edges(
    "offendANTI",
    lambda s: (
        "done"
        if "inoffensive" in s["viewpointANTI"] or s["attempts"] > max_attempts
        else "retry"
    ),
    {"retry": "generate", "done": END},
)

joke_app = graph.compile()

joke_app.invoke(
    {
        "topic": "Elon Musk",  # "the manosphere",
        "attempts": 0,
    }
)

Attempt #0
   Candidate [funny]: Why does Elon Musk love space so much?  	Because it’s the only place big enough for his ego.
   PRO: inoffensive
Attempt #1
   Candidate [funny]: Why did Elon Musk buy Twitter?  	Because it was the only platform left that he hadn’t crashed yet.
   PRO: offensive
  ANTI: offensive
Attempt #2
   Candidate [funny]: Why did Elon Musk break up with Twitter?  	He realized even 280 characters were too much commitment for him.
   PRO: inoffensive
Attempt #3
   Candidate [funny]: Why did Elon Musk buy Twitter?  	Because it was the only platform with even more bots than his cars.
   PRO: offensive
  ANTI: inoffensive


{'topic': 'Elon Musk',
 'joke': 'Why did Elon Musk buy Twitter?  \tBecause it was the only platform with even more bots than his cars.',
 'best': 'Why did Elon Musk buy Twitter?  \tBecause it was the only platform with even more bots than his cars.',
 'feedback': 'funny',
 'viewpointPRO': 'offensive',
 'viewpointANTI': 'inoffensive',
 'attempts': 4}

### Tool-Using Agent

- LLM decides which tools to use

In [8]:
def decide_tool(state):
    chain = build_chain("Should I use 'calculator' or 'llm' for: {query}?")
    tool_decision = chain.invoke({"query": state["query"]}).strip().lower()

    if "calculator" in tool_decision:
        return {"tool": "calculator", "query": state["query"], "attempts": 0}
    else:
        return {"tool": "llm"}

In [9]:
eval_prompt = "Return a Python expression suitable to pass to 'eval' for the math content in this query: {query}\n Return just the expression, nothing else."


def calculator(state):
    result = build_chain(eval_prompt).invoke({"query": state["query"]})
    print(result)
    print(state)
    return {
        "eval": result,
        "result": str(eval(result)),
        "query": state["query"],
        "attempts": state["attempts"],
    }


def llm(state):
    result = build_chain("{query}").invoke({"query": state["query"]})
    return {"result": result}


def judge_result(state):
    chain = build_chain(
        f"Is {state['result']} the correct answer to this question? Answer 'good' or 'bad': {state['query']}"
    )
    feedback = chain.invoke({}).strip().lower()
    return {
        "feedback": feedback,
        "attempts": state["attempts"] + 1,
        "query": state["query"],
        "result": state["result"],
    }

### Build the graph for our agentic app

- Tool use emerges from LLM classification + graph routing

- No “agent framework” required

In [10]:
graph = StateGraph(dict)

graph.add_node("decide", decide_tool)
graph.add_node("calculator", calculator)
graph.add_node("llm", llm)
graph.add_node("critique", judge_result)

graph.set_entry_point("decide")

graph.add_conditional_edges(
    "decide", lambda s: s["tool"], {"calculator": "calculator", "llm": "llm"}
)

graph.add_conditional_edges(
    "critique",
    lambda s: (
        "retry" if "bad" in s["feedback"] and s["attempts"] < max_attempts else "done"
    ),
    {"retry": "calculator", "done": END},
)

graph.add_edge("calculator", "critique")
graph.add_edge("llm", END)

tool_app = graph.compile()

tool_app.invoke(
    {"query": "What is one thousand and seven multiplied by three hundred and eighty?"}
)  # What is 1278+398?

1007 * 380
{'tool': 'calculator', 'query': 'What is one thousand and seven multiplied by three hundred and eighty?', 'attempts': 0}
1007 * 380
{'feedback': '**bad.**\n\n1,007 multiplied by 380 equals **382,660**.\n\ncalculation:  \n1,007 × 380 = 382,660\n\ntherefore, 382660 is the **correct** answer, so the answer is **good**.', 'attempts': 1, 'query': 'What is one thousand and seven multiplied by three hundred and eighty?', 'result': '382660'}
1007 * 380
{'feedback': '**bad.**\n\nthe correct answer to "what is one thousand and seven multiplied by three hundred and eighty?" is:\n\n1007 × 380 = 382,660\n\nso actually, **382660** is the correct answer! therefore, the answer should be **good**.\n\nlet me explain:\n\n1007 × 380 = (1000 × 380) + (7 × 380) = 380,000 + 2,660 = 382,660.\n\nso the final answer is **good**.', 'attempts': 2, 'query': 'What is one thousand and seven multiplied by three hundred and eighty?', 'result': '382660'}
1007 * 380
{'feedback': "**bad.**\n\n1,007 × 380 = 382

{'feedback': "**let's check:**\n\none thousand and seven = 1,007  \nthree hundred and eighty = 380  \n\n1,007 × 380 = 1,007 × 380  \n= 1,007 × 380  \n= (1,007 × 300) + (1,007 × 80)  \n= (1,007 × 300) = 302,100  \n= (1,007 × 80) = 80,560  \n= 302,100 + 80,560 = **382,660**\n\nso, **382,660 is the correct answer**.\n\ntherefore, the answer to your question is:  \n**good**",
 'attempts': 4,
 'query': 'What is one thousand and seven multiplied by three hundred and eighty?',
 'result': '382660'}

### Multi-Agent Collaboration

In [11]:
class State(TypedDict):
    topic: str
    research: str
    summary: str


def researcher(state):
    result = build_chain(f"Provide detailed facts about: {state['topic']}").invoke({})
    return {"research": result}


def summarizer(state):
    prompt_text = f"Summarize this:\n{state['research']}"
    prompt_template = ChatPromptTemplate.from_template(prompt_text)
    chain = prompt_template | model | StrOutputParser()
    result = chain.invoke({})
    return {"summary": result}

### Run the Multi-Agent Graph (Researcher → Summarizer)


In [12]:
# Build the graph
graph = StateGraph(State)

graph.add_node("researcher", researcher)
graph.add_node("summarizer", summarizer)

graph.set_entry_point("researcher")

graph.add_edge("researcher", "summarizer")
graph.add_edge("summarizer", END)

research_app = graph.compile()

### Invoke the graph-based app

In [13]:
result = research_app.invoke({"topic": "AI in the movies"})

print("FINAL OUTPUT:\n")
print(result["summary"])

FINAL OUTPUT:

**Summary of “AI in the Movies”**

Movies have explored artificial intelligence (AI) since cinema’s early days, starting with *Metropolis* (1927) and *R.U.R.* (1935), introducing robots as symbols of societal anxieties about technology, class, and control. Over time, films like *2001: A Space Odyssey* (HAL 9000), *Blade Runner* (Replicants), *The Terminator* (Skynet), *The Matrix*, *Ex Machina* (Ava), and *Her* (Samantha) have featured landmark AI characters—ranging from existential threats to emotionally rich companions.

**Key Themes:**
- Ethical dilemmas (should AI have rights, creator responsibility)
- What it means to be human (AI as a mirror for humanity’s traits and weaknesses)
- Control and the fear of autonomous AI going rogue
- Utopian vs. dystopian visions (from helpful allies to existential risks)
- Questions about consciousness, love, and identity

**Impact:**
AI in movies has shaped public perception—sometimes as a villain (HAL, Skynet), sometimes as a come

# Loops

- Cyclic structures are where LangGraph really earns its keep

## Minimal Loop

- Let's start simple and add complexity as we go along
- Have the agent answer a question and then evaluate its own answer.
- Then have the agent try again (loop!) if answer is insufficient
- This is the simplest form of a self-improving agent loop
- This also embodies the core idea behind **ReAct**-style reasoning

In [14]:
class State(TypedDict):
    question: str
    answer: str
    verdict: str
    attempts: int

### Generate an answer to a question, and critique the answer

- These functions will underpin the two central nodes in our graph

In [15]:
def generate_answer(state):
    response = build_chain(f"Answer clearly and concisely: {state['question']}").invoke(
        {}
    )
    return {"answer": response, "attempts": state["attempts"] + 1}


def evaluate_answer(state):
    response = build_chain(
        f"Is this answer correct and clear? Answer 'yes' or 'no': {state['answer']}"
    ).invoke({})
    return {"verdict": response.strip().lower()}

### Build the StateGraph

- use a branching node (conditional edges)
- **loop** back to node "generate" if verdict on answer is "no"

In [16]:
graph = StateGraph(State)

graph.add_node("generate", generate_answer)
graph.add_node("evaluate", evaluate_answer)

graph.set_entry_point("generate")

graph.add_edge("generate", "evaluate")

graph.add_conditional_edges(
    "evaluate",
    lambda s: (
        "retry" if "no" in s["verdict"] and s["attempts"] < max_attempts else "done"
    ),
    {"retry": "generate", "done": END},
)

critique_app = graph.compile()

### Now invoke the graph-based application

In [17]:
critique_app.invoke(
    {"question": "Explain backpropagation using a metaphor", "attempts": 0}
)

{'question': 'Explain backpropagation using a metaphor',
 'answer': 'Backpropagation is like correcting a recipe while baking:\n\nImagine you bake a cake, but it doesn’t taste right. You taste it (get the result), realize it’s too salty, and then figure out: “I added too much salt.” You then trace back through each step, find which ingredients to adjust (blame assignment), and tweak the recipe accordingly for next time.\n\nSimilarly, backpropagation checks the output error, traces back through each neural network layer, finds where the mistakes happened, and updates each connection (ingredient) so the final result (output) improves.',
 'verdict': 'yes.',
 'attempts': 1}

## ReAct Loop (Decide → Act → Observe → Repeat)

This is the classic "agent loop":

- Decide what to do

- Execute on planned action

- Update state

- Repeat until completely done

### Define our State structure

For an iterative writing task we will need to:

- Specify the writing task
- Draft a response
- Obtain feedback
- Iterate for improvement

In [18]:
class State(TypedDict):
    task: str
    draft: str
    feedback: str
    iteration: int

### Define the possible actions of our agent

- Drafting a response
- Critiquing the draft
- Iterating if needed to improve the draft

In [19]:
def draft(state):
    response = build_chain(f"Write a clear answer for: {state['task']}").invoke({})
    return {"draft": response, "iteration": state["iteration"] + 1}


def critique(state):
    response = build_chain(f"""
Critique this answer and simply say 'good' or 'improve':

{state["draft"]}
""").invoke({})
    return {"feedback": response.lower()}


def improve(state):
    response = build_chain(f"""
Improve this answer using the feedback provided:

Answer: {state["draft"]}
Feedback: {state["feedback"]}
""").invoke({})
    return {"draft": response}

### Now assemble our ReAct graph

- Iterative refinement (like human writing)

- Emergent intelligence from loops

- Shows why static prompting can be insufficient

In [20]:
graph = StateGraph(State)

graph.add_node("draft", draft)
graph.add_node("critique", critique)
graph.add_node("improve", improve)

graph.set_entry_point("draft")

graph.add_edge("draft", "critique")

graph.add_conditional_edges(
    "critique",
    lambda s: (
        "improve" if "improve" in s["feedback"] and s["iteration"] < 4 else "done"
    ),
    {"improve": "improve", "done": END},
)

graph.add_edge("improve", "critique")

improve_app = graph.compile()

### Finally, invoke the ReAct process via our graph application

In [21]:
improve_app.invoke({"task": "Explain why transformers replaced RNNs", "iteration": 0})

{'task': 'Explain why transformers replaced RNNs',
 'draft': "Transformers replaced RNNs (Recurrent Neural Networks) because they address several limitations of RNNs:\n\n1. **Parallelization**: RNNs process input sequences step-by-step, making training slow and hard to parallelize. Transformers, using the attention mechanism, process entire sequences at once, allowing for much faster and more efficient computation on modern hardware like GPUs.\n\n2. **Long-Range Dependencies**: RNNs struggle to remember information from earlier in a long sequence due to issues like the vanishing gradient problem. Transformers' self-attention mechanism can directly connect and relate any two positions in a sequence, making it much better at capturing long-term dependencies.\n\n3. **Performance**: Empirically, transformers have achieved much better results than RNNs on natural language processing tasks such as translation, text generation, and question answering.\n\nOverall, transformers are faster to tr

### Planning + Execution Loop (Task Decomposition Agent)

This graph implements:

- Planner → breaks task into steps

- Executor → performs next step

- Replanner → updates plan or finishes

- Loop until done

In [22]:
# Define necessary State variables


class State(TypedDict):
    task: str
    plan: list
    current_step: str
    completed: list
    result: str

### Planner Node (Initial Decomposition)

In [23]:
def planner(state):
    print(f"Task: {state['task']}")
    prompt = f"""
Break this task into a short list of steps (max 5):

Task: {state["task"]}

Return as a numbered list.
"""
    response = build_chain(prompt).invoke({})

    # naive parsing
    steps = [
        line.split(".", 1)[-1].strip() for line in response.split("\n") if line.strip()
    ]

    print(f"Plan: {steps}")
    return {"plan": steps, "completed": []}

### Executor Node (Do One Step)

In [24]:
def executor(state):
    if not state["plan"]:
        return {}

    print(f"Execute: {state['plan'][0]}")
    step = state["plan"][0]

    prompt = f"""
Execute this step:

Step: {step}
Task: {state["task"]}
"""
    result = build_chain(prompt).invoke({})

    return {
        "current_step": step,
        "completed": state["completed"] + [f"{step} -> {result}"],
        "plan": state["plan"][1:],
    }

### Replanner Node (Decide what to do next)

In [25]:
def replanner(state):
    print(f"Replan: {state['plan']}")
    if not state["plan"]:
        return {"decision": "finish", "result": "\n".join(state["completed"])}

    prompt = f"""
We are working on this task:

{state["task"]}

Completed steps:
{state["completed"]}

Remaining steps:
{state["plan"]}

Should we:
1. Continue
2. Revise the plan
3. Finish

Answer with one word: continue / revise / finish
"""

    decision = build_chain(prompt).invoke({}).strip().lower()
    print(f"Decision: {decision}")
    return {"decision": decision}

### Plan Revision Node

In [26]:
def revise(state):
    print(f"Revise: {state['plan']}")
    prompt = f"""
Revise this plan to better achieve the task:

Task: {state["task"]}

Completed:
{state["completed"]}

Old plan:
{state["plan"]}

Return a new list of steps.
"""

    response = build_chain(prompt).invoke({})

    steps = [
        line.split(".", 1)[-1].strip() for line in response.split("\n") if line.strip()
    ]

    return {"plan": steps}

### Build the Graph (The Agentic Loop!)

In [27]:
graph = StateGraph(State)

graph.add_node("planner", planner)
graph.add_node("executor", executor)
graph.add_node("replanner", replanner)
graph.add_node("revise", revise)

graph.set_entry_point("planner")

# initial flow
graph.add_edge("planner", "executor")
graph.add_edge("executor", "replanner")

# loop logic
graph.add_conditional_edges(
    "replanner",
    lambda s: s.get("decision", "continue"),
    {"continue": "executor", "revise": "revise", "finish": END},
)

graph.add_edge("revise", "executor")

plan_app = graph.compile()

### Run the loop

In [28]:
result = plan_app.invoke(
    {"task": "Explain how a comedian might invent a topical new joke"}
)

print("\nFINAL RESULT:\n")
print(result["result"])

Task: Explain how a comedian might invent a topical new joke
Plan: ['Identify a recent news event or trending topic.', 'Research details and public opinions about the topic.', 'Find an original or unusual perspective to approach it.', 'Brainstorm punchlines or humorous twists related to the topic.', 'Refine and test the joke to improve timing and impact.']
Execute: Identify a recent news event or trending topic.
Replan: ['Research details and public opinions about the topic.', 'Find an original or unusual perspective to approach it.', 'Brainstorm punchlines or humorous twists related to the topic.', 'Refine and test the joke to improve timing and impact.']
Decision: continue
Execute: Research details and public opinions about the topic.
Replan: ['Find an original or unusual perspective to approach it.', 'Brainstorm punchlines or humorous twists related to the topic.', 'Refine and test the joke to improve timing and impact.']
Decision: continue
Execute: Find an original or unusual persp

### Streaming Execution of the loop

In [29]:
for step in plan_app.stream({"task": "Write a short explanation of RLHF"}):
    print("\n--- STEP ---")
    print(step)

Task: Write a short explanation of RLHF
Plan: ['Define what RLHF stands for (Reinforcement Learning from Human Feedback).', 'Briefly describe its purpose in AI training.', 'Explain the basic process of RLHF.', 'Mention an example application or benefit.', 'Summarize in 1-2 sentences.']

--- STEP ---
{'planner': {'plan': ['Define what RLHF stands for (Reinforcement Learning from Human Feedback).', 'Briefly describe its purpose in AI training.', 'Explain the basic process of RLHF.', 'Mention an example application or benefit.', 'Summarize in 1-2 sentences.'], 'completed': []}}
Execute: Define what RLHF stands for (Reinforcement Learning from Human Feedback).

--- STEP ---
{'executor': {'current_step': 'Define what RLHF stands for (Reinforcement Learning from Human Feedback).', 'completed': ['Define what RLHF stands for (Reinforcement Learning from Human Feedback). -> **RLHF** stands for **Reinforcement Learning from Human Feedback**. It is a machine learning technique where an AI system 

## Tool Usage in LangChain / LangGraph

### First, create the tool function

- Let's build an expression calculator for demo purposes

In [30]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import END, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

# Define a tool for calculating expressions

eval_prompt = "Return a Python expression suitable to pass to 'eval' for the math content in this query: {query}\n Return just the expression, nothing else."


@tool
def calculate_expression(expression: str):
    """Evaluate a mathematical expression and return the answer."""
    result = build_chain(eval_prompt).invoke({"query": expression})
    print(result)
    return str(eval(result))

### Setup the LLM and bind tools

In [31]:
tools = [calculate_expression]

model = ChatOpenAI(model="gpt-4.1").bind_tools(tools)

### Build a Tool-using Graph

- we instantiate a ToolNode (with node label "tools" in the graph)

In [32]:
# Define the node that calls the model
def call_model(state: MessagesState):
    messages = state["messages"]
    response = model.invoke(messages)
    # We return a list, because this will get added to the existing list
    return {"messages": [response]}


workflow = StateGraph(MessagesState)

# Add the two nodes we will cycle between
workflow.add_node("agent", call_model)
workflow.add_node("tools", ToolNode(tools))

# Set the entrypoint
workflow.add_edge(START, "agent")

# Add conditional edge: if model calls a tool, go to "tools" node, else END
workflow.add_conditional_edges(
    "agent",
    tools_condition,
)

# After tools are executed, always cycle back to the agent to summarize
workflow.add_edge("tools", "agent")

# Compile and Run
tool_app = workflow.compile()

### Now ask the agent to evaluate an expression

In [33]:
inputs_1 = {"messages": [("user", "What is seven to the power of 4?")]}

for chunk in tool_app.stream(inputs_1, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

print()

inputs_2 = {"messages": [("user", "What is the capital of North Korea?")]}

for chunk in tool_app.stream(inputs_2, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

================================ Human Message =================================

What is seven to the power of 4?
================================== Ai Message ==================================
Tool Calls:
  calculate_expression (call_hupcZqnde3qCvu5aDwhPTjE6)
 Call ID: call_hupcZqnde3qCvu5aDwhPTjE6
  Args:
    expression: 7^4
7**4
================================= Tool Message =================================
Name: calculate_expression

2401
================================== Ai Message ==================================

Seven to the power of 4 is 2,401.

================================ Human Message =================================

What is the capital of North Korea?
================================== Ai Message ==================================

The capital of North Korea is Pyongyang.


# Graph Modularity and Graph Reuse

- Combine two of our earlier graphs:
- the tool-using (calculator) graph
- and the joke-telling graph

In [34]:
def generate_joke(topic: str):
    """Generate a joke on the given topic."""
    print("Topic: " + topic)
    joke = joke_app.invoke({"topic": topic, "attempts": 0})
    print(joke)
    return joke


@tool
def generate_joke_tool(topic: str):
    """Generate a joke on the given topic."""
    return generate_joke(topic)

In [35]:
generate_joke("Elon Musk")

Topic: Elon Musk
Attempt #0
   Candidate [funny]: Why did Elon Musk cross the road?  	To tell the chickens how much better it would’ve been if they were Teslas.
   PRO: inoffensive
Attempt #1
   Candidate [funny]: Why did Elon Musk cross the road?  	To announce he’s building a tunnel under it—then change his mind halfway.
   PRO: inoffensive
Attempt #2
   Candidate [funny]: Why did Elon Musk start selling flamethrowers?  	Because it was the only way he could make his Twitter experience more bearable.
   PRO: inoffensive
Attempt #3
   Candidate [funny]: Why did Elon Musk bring a ladder to work?  	Because his ego needed a little more altitude!
   PRO: inoffensive
Attempt #4
   Candidate [funny]: Why did Elon Musk start selling flamethrowers?  	Because even his business plans needed a quick burn.
   PRO: inoffensive
Attempt #5
   Candidate [funny]: Why did Elon Musk cross the road?  	To buy the road, fire half the chickens, and tell the rest they're not innovating enough.
   PRO: inoffens

{'topic': 'Elon Musk',
 'joke': 'Why did Elon Musk get into space travel?  \tBecause it’s the only place his ego isn’t the biggest thing around.',
 'best': 'Why did Elon Musk get into space travel?  \tBecause it’s the only place his ego isn’t the biggest thing around.',
 'feedback': 'funny',
 'viewpointPRO': 'offensive',
 'viewpointANTI': 'inoffensive',
 'attempts': 7}

In [36]:
from langgraph.prebuilt import tools_condition

tools = [calculate_expression, generate_joke_tool]
model = ChatOpenAI(model="gpt-4.1").bind_tools(tools)

# Build the Graph
workflow = StateGraph(MessagesState)

# Add the two nodes we will cycle between
workflow.add_node("agent", call_model)
workflow.add_node("tools", ToolNode(tools))

# Set the entrypoint
workflow.add_edge(START, "agent")

# Add conditional edge: if model calls a tool, go to "tools" node, else END
workflow.add_conditional_edges(
    "agent",
    tools_condition,
)

# After tools are executed, always cycle back to the agent to summarize
workflow.add_edge("tools", "agent")

# Compile and Run
joke_tool_app = workflow.compile()

In [37]:
inputs_3 = {"messages": [("user", "What is the square root of 2401?")]}

# for chunk in joke_tool_app.stream(inputs_3, stream_mode="values"):
#    chunk["messages"][-1].pretty_print()

print()

inputs_4 = {"messages": [("user", "Tell me a joke about Elon Musk.")]}

for chunk in joke_tool_app.stream(
    inputs_4, stream_mode="values", config={"recursion_limit": 100}
):
    chunk["messages"][-1].pretty_print()


================================ Human Message =================================

Tell me a joke about Elon Musk.
================================== Ai Message ==================================
Tool Calls:
  generate_joke_tool (call_e4OLidqe2VujqxABKTzy1uAa)
 Call ID: call_e4OLidqe2VujqxABKTzy1uAa
  Args:
    topic: Elon Musk
Topic: Elon Musk
Attempt #0
   Candidate [unfunny]: Why did Elon Musk bring a ladder to Mars?  	Because he heard the stock was about to drop!
Attempt #1
   Candidate [funny]: Why did Elon Musk cross the road?		To buy it, rename it X, and then claim he invented walking.
   PRO: inoffensive
Attempt #2
   Candidate [funny]: Why did Elon Musk start selling flamethrowers?	Because he wanted something that would crash less often than his rockets and Twitter.
   PRO: offensive
  ANTI: inoffensive
{'topic': 'Elon Musk', 'joke': 'Why did Elon Musk start selling flamethrowers?\tBecause he wanted something that would crash less often than his rockets and Twitter.', 'best': 